# ⚡ Electricity Utility Inefficiency & Residential Rate Analysis

> **Research Question:** Are system-level inefficiencies — high energy losses and poor load factors — statistically correlated with higher retail rates for residential consumers?

This notebook investigates a fundamental fairness question in the U.S. electricity sector: **do residential customers end up paying more when their utility operates inefficiently?**

Using the [CORGIS Electricity Dataset](https://corgis-edu.github.io/corgis/python/electricity/) — a structured derivative of the U.S. EIA Form 861 survey — the analysis engineers a set of efficiency and equity metrics, then examines their statistical relationships through a suite of interactive visualizations. New York State serves as the primary case study.

---
## 1. Setup
- `electricity` is the CORGIS data loader.
<br>
- `data_util` handles all data preparation and filtering. 
- `plot_util` contains every chart-building function.

In [14]:
import pandas as pd
import kaleido

import data.electricity as electricity

import src.util.plot_util as plot_util
import src.util.data_util as data_util

### 1.1 Data Loading & Metric Engineering

The raw dataset is a list of nested dictionaries — one record per utility. `prepare_data` engineers the following derived metrics from the raw columns:

| Metric | Formula | Interpretation |
|---|---|---|
| `ResidentialUnitPrice` | `(Residential Revenue / Sales) × 1000` | Residential rate in $/MWh |
| `IndustrialUnitPrice` | `Industrial Revenue / Industrial Sales` | Industrial rate in $/MWh |
| `IndustrialRevenueRatio` | `Industrial Revenue / Total Retail Revenue × 100` | % of a utility's revenue from industrial customers |
| `PriceSpread` | `ResidentialUnitPrice − IndustrialUnitPrice` | Pricing gap between Residential and Industrial customers |
| `SystemLossPercentage` | `(Uses.Losses / Sources.Total) × 100` | Energy lost in transmission/distribution as % of supply |
| `LoadFactor` | `Sources.Total / (Summer Peak Demand × 8760)` | Utilization efficiency — how fully the system capacity is used |
| `FairnessIndex` | `ResidentialUnitPrice / LoadFactor` | Composite burden: residential price adjusted for system efficiency |

<br>

**Note on LoadFactor:**
 A value of 1.0 means the utility delivers energy at exactly its peak demand capacity around the clock — theoretically perfect utilization. Real utilities typically range from 0.3–0.7. A low load factor suggests underused infrastructure whose fixed costs are spread across fewer delivered MWh.

In [15]:
# Retrieve entire US-based dataset
utility = electricity.get_utility()
df = pd.json_normalize(utility)

# Limit to the state of New York and relevant columns
ny_df = data_util.get_state_data('NY', df)

# Add key metrics for use in later plots
ny_df = data_util.prepare_data(ny_df)

---
## 2. State Selection — Why New York?

Rather than choosing a state arbitrarily, we use `get_state_variance` to rank all 50 states across five dimensions that make for a rich case study:

|  | Why it matters |
|---|---|
| **# Utilities** | More utilities = more data points and statistical power |
| **# Utility Types** | More ownership models = richer segmentation and comparison |
| **Residential Price Std. Dev.** | High spread = real price variation to explain, not a flat market |
| **Max System Loss %** | Outlier utilities = compelling examples for the core hypothesis |
| **Industrial Revenue %** | High industrial dependency = cross-subsidization dynamics worth examining |

The table below highlights the top 10 states by residential price variance. **New York (highlighted)** ranks highly because it uniquely satisfies *all five* criteria simultaneously.

In [16]:
state_variance = data_util.get_state_variance(data_util.prepare_data(df))

top_variance_table = plot_util.get_state_variance_table(state_variance)
# top_variance_table.show()

### Why New York specifically?

Beyond the quantitative ranking, New York offers several structural advantages for this analysis:

- **7 distinct utility ownership types** — more than almost any other state, enabling meaningful cross-model comparisons within a single regulatory environment.
- **100+ individual utilities** — sufficient sample size to draw statistically meaningful conclusions.
- **Extreme geographic diversity** — Con Edison serves dense urban Manhattan while small rural cooperatives serve upstate communities. This natural variation in operating context creates a richer distribution of efficiency and pricing outcomes than a homogeneous state would.
- **High system loss outliers** — NY's transmission infrastructure includes some of the oldest urban grid segments in the country, producing the high-loss edge cases that make the core hypothesis testable.
- **Active regulatory environment** — the NY Public Service Commission is one of the most scrutinized utility regulators in the U.S., meaning this analysis connects directly to real-world policy questions.

> **Interpretation:** The highlighted row confirms NY's selection. High residential price variance signals there is real variation to explain — a state where all utilities charge similar rates would offer nothing for this analysis to find.

---
## 3. Ownership Model Analysis

Before testing the efficiency hypothesis, we establish baseline pricing distributions by ownership model. This matters because ownership type is a structural variable — investor-owned utilities (IOUs) answer to shareholders and profit motives, while cooperatives and municipals serve member-owners or public ratepayers. If ownership type already explains most of the price variance, efficiency metrics may be redundant.

In [17]:
# Keep utilities that offer residential services
residential_customers_df = data_util.get_customer_utilities(
    ny_df,  "Residential").round(2)
res_boxplot = plot_util.get_utility_type_box_plot(
    residential_customers_df, "Residential")

# res_boxplot.show()

# Keep utilities that offer industrial services
industrial_customers_df = data_util.get_customer_utilities(
    ny_df, "Industrial").round(2)
ind_boxplot = plot_util.get_utility_type_box_plot(
    industrial_customers_df, "Industrial")

# ind_boxplot.show()

**Interpretation:** Comparing the residential and industrial plots side-by-side reveals whether different ownership models systematically favor one customer class over the other. A utility type showing high residential rates but low industrial rates is a candidate for cross-subsidization — where industrial discounts are effectively funded by residential ratepayers.

---
## 4. Correlation Analysis

Before building directional charts, we compute a Pearson correlation matrix across all five key engineered metrics. This serves two purposes: it quantifies the *strength and direction* of every pairwise relationship, and it surfaces any unexpected correlations that warrant further investigation. A positive correlation between `SystemLossPercentage` and `ResidentialUnitPrice` would support the hypothesis. A negative correlation between `LoadFactor` and `ResidentialUnitPrice` (higher efficiency → lower price) would also support it. The `FairnessIndex` row shows the composite signal.

In [18]:
# Key metrics: System Loss %, Load Factor', Industrial Revenue %, Price Spread, FairnessIndex
heatmap = plot_util.get_key_metrics_corr_matrix(ny_df)

# heatmap.show()

**Interpretation:** The correlation matrix provides the statistical foundation for the charts that follow. Moderate-to-strong correlations in the expected direction — particularly between `SystemLossPercentage`/`LoadFactor` and `ResidentialUnitPrice` — validate the fairness audit as a meaningful exercise rather than speculation.

---
## 5. Fairness Audit — Efficiency vs. Residential Price

This is the central test of the research question, presented as two side-by-side scatter plots sharing a y-axis (residential price). Each plot approaches inefficiency from a different angle:

| Plot | X-axis | What it tests |
|---|---|---|
| Left | `SystemLossPercentage` | Does wasted energy in the grid cost residential customers more? |
| Right | `LoadFactor` | Does underutilized infrastructure translate to higher per-MWh costs? |

> **Filtering note:** This chart excludes utilities with zero system loss or zero load factor. These edge cases typically represent pass-through entities (pure resellers) or data reporting anomalies — including them would distort the OLS fit.

In [19]:
# Keep residential utilities that are using energy (instead of ONLY reseale, etc.)
residential_lf_sys_loss_df = data_util.get_residential_sys_loss(ny_df)
residential_lf_sys_loss_df = data_util.get_residential_load_factor(
    residential_lf_sys_loss_df).round(2)

dual_y_scatter = plot_util.get_fairness_dual_y_scatter_plot(
    residential_lf_sys_loss_df)

dual_y_scatter.show()

**Interpretation:** If the left plot shows an upward slope and the right plot shows a downward slope — both with meaningful R² values — it supports the hypothesis that operational inefficiency is a statistically significant predictor of residential rate levels. Ownership model clustering (color groupings) indicates how much of the relationship is mediated by structural type rather than efficiency alone.

---
## 6. Rate Disparity — Residential vs. Industrial

Even if inefficiency drives prices up overall, the burden may not fall equally. This chart examines the **top 10 utilities by PriceSpread** — the gap between what residential and industrial customers pay per MWh. Industrial customers typically negotiate volume discounts — some spread is expected. But when residential customers pay 2–3× the industrial rate at the same utility, it raises questions about whether the rate structure reflects true cost-of-service differences or something else. This chart identifies the specific utilities where that premium is most extreme.

In [20]:
# Keep utilities that offer both industrial and residential services
res_ind_customers_df = data_util.get_customer_utilities(
    ny_df, "Industrial").round(2)

dumbbell = plot_util.get_rate_disparity_dumbbell_plot(res_ind_customers_df)

# dumbbell.show()

**Interpretation:** Cross-reference the utilities appearing here with their ownership type and load factor from earlier charts. Utilities that appear in both this chart (high spread) *and* the fairness audit (high system loss or low load factor) are the most analytically interesting cases — they are simultaneously inefficient *and* passing a disproportionate share of costs to residential customers.

---
## 7. Energy Flow Analysis

The Sankey diagrams provide operational context for the efficiency metrics computed above. Rather than a single number, they show the *full picture* of where a utility's energy comes from and where it goes. A wide Losses band at a utility with high residential prices is the inefficiency–cost story in one image. Compare the utility-level Sankey against the U.S. national average to understand whether NY's profile is typical or anomalous.

In [21]:
# Look at a specific utility's energy usage/flow
utility_usage = data_util.get_utility_usage(ny_df.sum(), level="State")
ny_sankey = plot_util.get_energy_use_sankey_plot(utility_usage)

# ny_sankey.show()

# Look at the entire country's energy usage/flow
us_energy_flow = df.groupby(["Utility.State"]).sum().sum()[1:]
us_energy_flow = data_util.get_utility_usage(us_energy_flow, level="US")

us_sankey = plot_util.get_energy_use_sankey_plot(us_energy_flow)

# us_sankey.show()

**Interpretation:** Comparing an individual utility's Losses band against the U.S. national benchmark gives immediate context — is this utility's loss rate typical, or a meaningful outlier? A utility with a Losses band notably wider than the national average, combined with high residential prices from the fairness audit, provides the clearest evidence for the core hypothesis.

### 7b. Interactive Explorer — Individual Utility Flow

In [ ]:
# Sankey chart with a drop down energy flow for each individual utility

# plot_util.add_utility_dropdown(ny_sankey, df=ny_df)

---
## 8. Export

All charts are exported as SVGs to the `/images` directory for use in the README, portfolio writeup, and presentations. SVG format preserves full resolution at any display size.

> **Requires:** `kaleido` module

In [23]:
%pip install kaleido

# Gather all plots and export them as SVGs
plots = [top_variance_table, res_boxplot, ind_boxplot, heatmap,
         dual_y_scatter, dumbbell, ny_sankey, us_sankey]

plot_util.export_plots_as_svg(plots)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
